#  ⭐ `dim_duracao`


In [1]:
%run ../../config/bootstrap.py


from pathlib import Path
from utils import get_project_root , normalize_duracao

import re
import pandas as pd
from unidecode import unidecode

In [2]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [3]:
path = Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos.parquet"
bronze = pd.read_parquet(Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos.parquet")  
bronze['DURACAO']= bronze['DURACAO'].fillna('DESCONHECIDO')
bronze = (
    bronze[['DURACAO']]
    .value_counts()
    .rename('FREQUENCIA_REGISTROS')
    .reset_index()
) 
bronze.columns

Index(['DURACAO', 'FREQUENCIA_REGISTROS'], dtype='object')

In [4]:
bronze.head()

,DURACAO,FREQUENCIA_REGISTROS
0,DESCONHECIDO,473049
1,1 dia,20933
2,2 dia,4107
3,3 dia,3079
4,1 hora,2959


In [5]:
bronze.DURACAO.nunique()

1538

In [6]:
bronze.to_csv(Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos_DURACAO.csv", sep=";", index=False)

# 🥈 Silver

normalized data, medium quality


### dev

In [7]:
silver = bronze.copy()
 
silver.columns

Index(['DURACAO', 'FREQUENCIA_REGISTROS'], dtype='object')

In [8]:
silver.head(40)

,DURACAO,FREQUENCIA_REGISTROS
0,DESCONHECIDO,473049
1,1 dia,20933
2,2 dia,4107
3,3 dia,3079
4,1 hora,2959
5,5 dia,2485
6,1 minuto,2274
7,4 dia,2253
8,30 minuto,2101
9,7 dia,1878


In [9]:
hist = normalize_duracao(bronze, coluna="DURACAO")

In [10]:
hist.head()

,DURACAO,FREQUENCIA_REGISTROS,DURACAO_TIPO_CHAVE,DURACAO_TIPO_VALOR,DURACAO_VALOR
0,DESCONHECIDO,473049,desconhecido,0,NaN
1,1 dia,20933,dia,4,1.0
2,2 dia,4107,dia,4,2.0
3,3 dia,3079,dia,4,3.0
4,1 hora,2959,hora,3,1.0


In [11]:
(
    hist[['DURACAO_TIPO_CHAVE', 'DURACAO_TIPO_VALOR']]
    .value_counts(dropna=False)
    .rename("count")
    .reset_index()
    .sort_values("DURACAO_TIPO_VALOR")
)

,DURACAO_TIPO_CHAVE,DURACAO_TIPO_VALOR,count
7,desconhecido,0,18
5,segundo,1,29
1,minuto,2,215
2,hora,3,116
0,dia,4,984
6,semana,5,28
3,mes,6,85
4,ano,7,63


In [11]:
hist[['DURACAO_VALOR']].value_counts(dropna=False)

DURACAO_VALOR
NaN              18
2.000            12
1.000            12
3.000            12
7.000            11
6.000            11
5.000            11
4.000            11
9.000            10
8.000            10
0.000            10
10.000            8
40.000            7
24.000            7
20.000            7
18.000            7
15.000            7
36.000            6
33.000            6
30.000            6
25.000            6
19.000            6
16.000            6
14.000            6
12.000            6
11.000            6
13.000            6
21.000            5
60.000            5
38.000            5
45.000            5
17.000            5
23.000            5
26.000            5
28.000            5
22.000            5
35.000            5
34.000            5
31.000            4
90.000            4
46.000            4
70.000            4
50.000            4
51.000            4
68.000            4
74.000            4
42.000            4
180.000           4
1.500             4
2.500 

In [12]:
hist.to_parquet(Path(project_root) / "data/02_silver/hist_dim_duracao/hist_dim_duracao.parquet", index=False)

# 🥇 Gold


In [13]:
hist.columns

Index(['DURACAO', 'FREQUENCIA_REGISTROS', 'DURACAO_TIPO_CHAVE',
       'DURACAO_TIPO_VALOR', 'DURACAO_VALOR'],
      dtype='object')

In [14]:
dim = hist[['DURACAO_TIPO_CHAVE', 'DURACAO_TIPO_VALOR']].drop_duplicates().sort_values('DURACAO_TIPO_VALOR')
dim

,DURACAO_TIPO_CHAVE,DURACAO_TIPO_VALOR
0,desconhecido,0
75,segundo,1
6,minuto,2
4,hora,3
1,dia,4
63,semana,5
28,mes,6
25,ano,7


In [15]:
dim.to_parquet(Path(project_root) / "data/03_gold/dim_duracao/dim_duracao.parquet", index=False)